# ReadmitIQ: Clinical Transformer Fine-Tuning (Colab GPU Accelerated)

This notebook fine-tunes **DistilBERT** (`distilbert-base-uncased`) on synthetic clinical discharge notes to predict 30-day hospital readmission risk.

### Instructions for Morning Execution:
1. **Runtime Type**: In Colab menu, select **Runtime -> Change runtime type -> T4 GPU**.
2. **Files to Upload**:
   - Upload `data/processed/train_rag.parquet` (69,535 rows)
   - Upload `data/processed/val_rag.parquet` (14,902 rows)
3. **Run All Cells**: Takes ~12-15 minutes on a free T4 GPU.
4. **Outputs to Copy Back** into local repo:
   - Copy `distilbert_val_probs.npy` to `reports/colab/distilbert_val_probs.npy`
   - Copy `distilbert_val_metrics.json` to `reports/colab/distilbert_val_metrics.json`
   - (Optional) Copy `distilbert_readmit_model.zip` to `models/distilbert_readmit_model.zip`
5. Then run: `python src/transformer_import.py` to fold results into the ablation table.

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install Required Libraries
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve, brier_score_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Training Device: {device}")
assert device.type == "cuda", "Please enable GPU in Runtime -> Change runtime type -> T4 GPU!"

In [ ]:
# Step 3: Load Train and Validation Parquets
train_df = pd.read_parquet("train_rag.parquet", columns=["clinical_note", "readmitted_30d"])
val_df = pd.read_parquet("val_rag.parquet", columns=["clinical_note", "readmitted_30d"])

print(f"Loaded Train notes: {len(train_df):,}, Val notes: {len(val_df):,}")

# Optional stratified subset if desired to cap at 30,000 notes for ultra-fast training (~6 min)
TRAIN_SUBSET_SIZE = 30000  # Set to None or len(train_df) for full 69k training
if TRAIN_SUBSET_SIZE and len(train_df) > TRAIN_SUBSET_SIZE:
    pos = train_df[train_df["readmitted_30d"] == 1]
    neg = train_df[train_df["readmitted_30d"] == 0]
    pos_n = int(TRAIN_SUBSET_SIZE * 0.114)
    neg_n = TRAIN_SUBSET_SIZE - pos_n
    train_df = pd.concat([
        pos.sample(n=pos_n, random_state=42),
        neg.sample(n=neg_n, random_state=42)
    ]).sample(frac=1.0, random_state=42).reset_index(drop=True)
    print(f"Subsampled train cohort to {len(train_df):,} stratified notes (prevalence: {train_df['readmitted_30d'].mean():.2%})")
else:
    print(f"Using full train cohort of {len(train_df):,} notes.")

print(f"Evaluating on FULL validation cohort: {len(val_df):,} notes (no subsampling).")

In [ ]:
# Step 4: Tokenization Dataset
class ClinicalNotesDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ClinicalNotesDataset(train_df["clinical_note"], train_df["readmitted_30d"], tokenizer)
val_dataset = ClinicalNotesDataset(val_df["clinical_note"], val_df["readmitted_30d"], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
# Step 5: Initialize Model with Weighted Loss & FP16 Optimizer
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# 7.7x weight on positive class (approx 11.4% prevalence)
class_weights = torch.tensor([1.0, 7.7], dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

epochs = 2
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=True)

In [ ]:
# Step 6: Mixed-Precision Training Loop
for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_train_loss += loss.item()
        if (step + 1) % 150 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Step {step+1}/{len(train_loader)} | Current Loss: {loss.item():.4f}")

    print(f"--> Completed Epoch {epoch+1} | Mean Loss: {total_train_loss / len(train_loader):.4f}")

In [ ]:
# Step 7: Evaluate on the FULL Validation Split (14,902 notes)
model.eval()
val_probs = []
val_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits.float(), dim=-1)[:, 1]
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(batch["label"].numpy())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

auc = float(roc_auc_score(val_labels, val_probs))
brier = float(brier_score_loss(val_labels, val_probs))

# 1,000 Bootstrap 95% Confidence Interval
rng = np.random.RandomState(42)
boot_aucs = []
n_val = len(val_labels)
for _ in range(1000):
    idx = rng.randint(0, n_val, n_val)
    if len(np.unique(val_labels[idx])) < 2:
        continue
    boot_aucs.append(roc_auc_score(val_labels[idx], val_probs[idx]))

ci_lower = float(np.percentile(boot_aucs, 2.5))
ci_upper = float(np.percentile(boot_aucs, 97.5))

# Precision@top-10% and Lift
k_10 = max(1, int(len(val_labels) * 0.10))
top_10_idx = np.argsort(val_probs)[::-1][:k_10]
prec_top_10 = float(np.mean(val_labels[top_10_idx]))
lift = float(prec_top_10 / np.mean(val_labels))

# Optimal threshold for max F1
precs, recs, ths = precision_recall_curve(val_labels, val_probs)
denom = precs + recs
f1s = np.divide(2 * precs * recs, denom, out=np.zeros_like(precs), where=(denom > 0))
best_idx = int(np.argmax(f1s))
best_th = float(ths[best_idx]) if best_idx < len(ths) else 0.5
f1_val = float(f1s[best_idx])

metrics = {
    "model": "DistilBERT (Clinical Notes)",
    "rag_feature": False,
    "auc_roc": round(auc, 4),
    "auc_ci_95": [round(ci_lower, 4), round(ci_upper, 4)],
    "optimal_threshold": round(best_th, 4),
    "f1": round(f1_val, 4),
    "precision_top_10pct": round(prec_top_10, 4),
    "lift_top_10pct": round(lift, 2),
    "brier_score": round(brier, 4),
    "val_samples": len(val_labels)
}

print("=" * 60)
print(f"DistilBERT Validation AUC: {auc:.4f} [95% CI: {ci_lower:.4f} - {ci_upper:.4f}]")
print(f"Precision@top-10%: {prec_top_10:.4f} | Lift: {lift:.2f}x | F1: {f1_val:.4f}")
print(f"Brier Score: {brier:.4f}")
print("=" * 60)

In [ ]:
# Step 8: Export Validation Probabilities and Metrics for Local Repo
np.save("distilbert_val_probs.npy", val_probs)
with open("distilbert_val_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Optional: Save model weights
model.save_pretrained("distilbert_readmit_model")
tokenizer.save_pretrained("distilbert_readmit_model")
!zip -q -r distilbert_readmit_model.zip distilbert_readmit_model

print("Successfully exported:")
print("1. distilbert_val_probs.npy")
print("2. distilbert_val_metrics.json")
print("3. distilbert_readmit_model.zip")
print("Copy these files to reports/colab/ in your local project repo.")